In [2]:
import pandas as pd
import numpy as np 

In [3]:
season = 2025

url = f"https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{season}.parquet"

pbp = pd.read_parquet(url)

In [4]:
pbp.shape

(48771, 372)

In [5]:
pbp.head()

,play_id,game_id,old_game_id,home_team,away_team,season_type,week,posteam,posteam_type,defteam,...,out_of_bounds,home_opening_kickoff,qb_epa,xyac_epa,xyac_mean_yardage,xyac_median_yardage,xyac_success,xyac_fd,xpass,pass_oe
0,1.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,NaN,NaN,NaN,...,0.0,0.0,-0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,40.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,0.0,-0.352700,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,63.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,0.0,-0.190052,NaN,NaN,NaN,NaN,NaN,0.511128,-51.112807
3,85.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,1.0,0.0,1.317340,0.939998,4.750889,3.0,0.666726,0.43911,0.668940,33.105969
4,115.0,2025_01_ARI_NO,2025090705,NO,ARI,REG,1,ARI,away,NO,...,0.0,0.0,-1.694360,NaN,NaN,NaN,NaN,NaN,0.492038,50.796208


In [6]:
[x for x in pbp.columns if any(keyword in x.lower() for keyword in [
    "receiver",
    "rusher",
    "yardline",
    "air_yards",
    "target",
    "touchdown",
    "fantasy"
])]

['yardline_100',
 'air_yards',
 'touchdown',
 'pass_touchdown',
 'rush_touchdown',
 'return_touchdown',
 'receiver_player_id',
 'receiver_player_name',
 'rusher_player_id',
 'rusher_player_name',
 'lateral_receiver_player_id',
 'lateral_receiver_player_name',
 'lateral_rusher_player_id',
 'lateral_rusher_player_name',
 'rusher',
 'rusher_jersey_number',
 'receiver',
 'receiver_jersey_number',
 'rusher_id',
 'receiver_id',
 'fantasy_player_name',
 'fantasy_player_id',
 'fantasy',
 'fantasy_id']

In [7]:
pbp["play_type"].value_counts(dropna=False).head(15)

play_type
pass           19737
run            14895
no_play         4723
kickoff         2927
punt            2042
NaN             1445
extra_point     1330
field_goal      1140
qb_kneel         452
qb_spike          80
Name: count, dtype: int64

In [8]:
offense = pbp[pbp["play_type"].isin(["pass", "run"])].copy()

offense.shape

(34632, 372)

In [9]:
offense[
    [
        "play_type",
        "yardline_100",
        "air_yards",
        "receiver_player_name",
        "rusher_player_name",
        "touchdown"
    ]
].head(20)

,play_type,yardline_100,air_yards,receiver_player_name,rusher_player_name,touchdown
2,run,78.0,NaN,NaN,J.Conner,0.0
3,pass,75.0,3.0,T.McBride,NaN,0.0
4,pass,64.0,NaN,NaN,NaN,0.0
5,run,75.0,NaN,NaN,J.Conner,0.0
6,run,77.0,NaN,NaN,T.Benson,0.0
8,run,77.0,NaN,NaN,A.Kamara,0.0
9,pass,74.0,2.0,R.Shaheed,NaN,0.0
10,pass,74.0,14.0,NaN,NaN,0.0
12,pass,66.0,-3.0,T.McBride,NaN,0.0
13,run,61.0,NaN,NaN,K.Murray,0.0


In [10]:
[x for x in pbp.columns if x in [
    "complete_pass",
    "incomplete_pass",
    "pass_attempt",
    "rush_attempt",
    "sack"
]]

['incomplete_pass', 'rush_attempt', 'pass_attempt', 'sack', 'complete_pass']

In [11]:
opportunities = offense[
    (offense["rush_attempt"] == 1)
    | (offense["receiver_player_id"].notna())
].copy()

opportunities.shape

(32477, 372)

In [12]:
opportunities["player_name"] = np.where(
    opportunities["rush_attempt"] == 1,
    opportunities["rusher_player_name"],
    opportunities["receiver_player_name"]
)

opportunities[
    ["play_type", "player_name", "yardline_100", "air_yards"]
].head(15)

,play_type,player_name,yardline_100,air_yards
2,run,J.Conner,78.0,NaN
3,pass,T.McBride,75.0,3.0
5,run,J.Conner,75.0,NaN
6,run,T.Benson,77.0,NaN
8,run,A.Kamara,77.0,NaN
9,pass,R.Shaheed,74.0,2.0
12,pass,T.McBride,66.0,-3.0
13,run,K.Murray,61.0,NaN
14,run,J.Conner,48.0,NaN
19,run,J.Conner,39.0,NaN


In [13]:
opportunities["player_id"] = np.where(
    opportunities["rush_attempt"] == 1,
    opportunities["rusher_player_id"],
    opportunities["receiver_player_id"]
)

opportunities[
    ["play_type", "player_name", "player_id"]
].head(15)

,play_type,player_name,player_id
2,run,J.Conner,00-0033553
3,pass,T.McBride,00-0037744
5,run,J.Conner,00-0033553
6,run,T.Benson,00-0039921
8,run,A.Kamara,00-0033906
9,pass,R.Shaheed,00-0037545
12,pass,T.McBride,00-0037744
13,run,K.Murray,00-0035228
14,run,J.Conner,00-0033553
19,run,J.Conner,00-0033553


In [14]:
opportunities["red_zone"] = (opportunities["yardline_100"] <= 20).astype(int)

opportunities["goal_line"] = (opportunities["yardline_100"] <= 5).astype(int)

opportunities[
    ["player_name", "yardline_100", "red_zone", "goal_line"]
].head(20)

,player_name,yardline_100,red_zone,goal_line
2,J.Conner,78.0,0,0
3,T.McBride,75.0,0,0
5,J.Conner,75.0,0,0
6,T.Benson,77.0,0,0
8,A.Kamara,77.0,0,0
9,R.Shaheed,74.0,0,0
12,T.McBride,66.0,0,0
13,K.Murray,61.0,0,0
14,J.Conner,48.0,0,0
19,J.Conner,39.0,0,0


In [15]:
[x for x in pbp.columns if any(keyword in x.lower() for keyword in [
    "receiving_yards",
    "rushing_yards",
    "complete_pass",
    "fumble",
    "two_point"
])]

['two_point_conv_result',
 'two_point_conversion_prob',
 'incomplete_pass',
 'fumble_forced',
 'fumble_not_forced',
 'fumble_out_of_bounds',
 'fumble_lost',
 'two_point_attempt',
 'fumble',
 'complete_pass',
 'receiving_yards',
 'rushing_yards',
 'lateral_receiving_yards',
 'lateral_rushing_yards',
 'forced_fumble_player_1_team',
 'forced_fumble_player_1_player_id',
 'forced_fumble_player_1_player_name',
 'forced_fumble_player_2_team',
 'forced_fumble_player_2_player_id',
 'forced_fumble_player_2_player_name',
 'fumbled_1_team',
 'fumbled_1_player_id',
 'fumbled_1_player_name',
 'fumbled_2_player_id',
 'fumbled_2_player_name',
 'fumbled_2_team',
 'fumble_recovery_1_team',
 'fumble_recovery_1_yards',
 'fumble_recovery_1_player_id',
 'fumble_recovery_1_player_name',
 'fumble_recovery_2_team',
 'fumble_recovery_2_yards',
 'fumble_recovery_2_player_id',
 'fumble_recovery_2_player_name',
 'defensive_two_point_attempt',
 'defensive_two_point_conv']

In [16]:
scoring_columns = [
    "complete_pass",
    "receiving_yards",
    "rushing_yards",
    "rush_touchdown",
    "pass_touchdown",
    "fumble_lost",
    "two_point_attempt",
    "two_point_conv_result"
]

[col for col in scoring_columns if col in pbp.columns]

['complete_pass',
 'receiving_yards',
 'rushing_yards',
 'rush_touchdown',
 'pass_touchdown',
 'fumble_lost',
 'two_point_attempt',
 'two_point_conv_result']

In [17]:
opportunities["actual_fp"] = (
    opportunities["complete_pass"].fillna(0) * 1.0
    + opportunities["receiving_yards"].fillna(0) * 0.1
    + opportunities["rushing_yards"].fillna(0) * 0.1
    + opportunities["touchdown"].fillna(0) * 6.0
)

opportunities[
    [
        "player_name",
        "play_type",
        "complete_pass",
        "receiving_yards",
        "rushing_yards",
        "touchdown",
        "actual_fp"
    ]
].head(20)

,player_name,play_type,complete_pass,receiving_yards,rushing_yards,touchdown,actual_fp
2,J.Conner,run,0.0,NaN,3.0,0.0,0.3
3,T.McBride,pass,1.0,11.0,NaN,0.0,2.1
5,J.Conner,run,0.0,NaN,-2.0,0.0,-0.2
6,T.Benson,run,0.0,NaN,1.0,0.0,0.1
8,A.Kamara,run,0.0,NaN,3.0,0.0,0.3
9,R.Shaheed,pass,0.0,NaN,NaN,0.0,0.0
12,T.McBride,pass,1.0,5.0,NaN,0.0,1.5
13,K.Murray,run,0.0,NaN,13.0,0.0,1.3
14,J.Conner,run,0.0,NaN,1.0,0.0,0.1
19,J.Conner,run,0.0,NaN,12.0,0.0,1.2


In [18]:
model_preview = opportunities[
    [
        "player_name",
        "play_type",
        "yardline_100",
        "air_yards",
        "red_zone",
        "goal_line",
        "rush_attempt",
        "complete_pass",
        "actual_fp"
    ]
].copy()

model_preview.head(20)

,player_name,play_type,yardline_100,air_yards,red_zone,goal_line,rush_attempt,complete_pass,actual_fp
2,J.Conner,run,78.0,NaN,0,0,1.0,0.0,0.3
3,T.McBride,pass,75.0,3.0,0,0,0.0,1.0,2.1
5,J.Conner,run,75.0,NaN,0,0,1.0,0.0,-0.2
6,T.Benson,run,77.0,NaN,0,0,1.0,0.0,0.1
8,A.Kamara,run,77.0,NaN,0,0,1.0,0.0,0.3
9,R.Shaheed,pass,74.0,2.0,0,0,0.0,0.0,0.0
12,T.McBride,pass,66.0,-3.0,0,0,0.0,1.0,1.5
13,K.Murray,run,61.0,NaN,0,0,1.0,0.0,1.3
14,J.Conner,run,48.0,NaN,0,0,1.0,0.0,0.1
19,J.Conner,run,39.0,NaN,0,0,1.0,0.0,1.2


In [19]:
opportunities.to_parquet(
    "../../data/processed_2025_opportunities.parquet",
    index=False
)

In [20]:
test_load = pd.read_parquet(
    "../../data/processed_2025_opportunities.parquet"
)

test_load.shape

(32477, 377)

In [21]:
test_load[
    [
        "player_name",
        "player_id",
        "red_zone",
        "goal_line",
        "actual_fp"
    ]
].head()

,player_name,player_id,red_zone,goal_line,actual_fp
0,J.Conner,00-0033553,0,0,0.3
1,T.McBride,00-0037744,0,0,2.1
2,J.Conner,00-0033553,0,0,-0.2
3,T.Benson,00-0039921,0,0,0.1
4,A.Kamara,00-0033906,0,0,0.3


In [22]:
import pandas as pd
import numpy as np

opportunities = pd.read_parquet(
    "../../data/processed_2025_opportunities.parquet"
)

opportunities.shape

(32477, 377)

In [23]:
opportunities["target"] = (
    opportunities["receiver_player_id"].notna()
).astype(int)

opportunities[
    ["play_type", "player_name", "rush_attempt", "target"]
].head(15)

,play_type,player_name,rush_attempt,target
0,run,J.Conner,1.0,0
1,pass,T.McBride,0.0,1
2,run,J.Conner,1.0,0
3,run,T.Benson,1.0,0
4,run,A.Kamara,1.0,0
5,pass,R.Shaheed,0.0,1
6,pass,T.McBride,0.0,1
7,run,K.Murray,1.0,0
8,run,J.Conner,1.0,0
9,run,J.Conner,1.0,0


In [24]:
[x for x in opportunities.columns if x in [
    "posteam",
    "defteam",
    "week",
    "game_id"
]]

['game_id', 'week', 'posteam', 'defteam']

In [25]:
opportunities[
    ["game_id", "week", "posteam", "defteam", "player_name", "target", "rush_attempt"]
].head(15)

,game_id,week,posteam,defteam,player_name,target,rush_attempt
0,2025_01_ARI_NO,1,ARI,NO,J.Conner,0,1.0
1,2025_01_ARI_NO,1,ARI,NO,T.McBride,1,0.0
2,2025_01_ARI_NO,1,ARI,NO,J.Conner,0,1.0
3,2025_01_ARI_NO,1,ARI,NO,T.Benson,0,1.0
4,2025_01_ARI_NO,1,NO,ARI,A.Kamara,0,1.0
5,2025_01_ARI_NO,1,NO,ARI,R.Shaheed,1,0.0
6,2025_01_ARI_NO,1,ARI,NO,T.McBride,1,0.0
7,2025_01_ARI_NO,1,ARI,NO,K.Murray,0,1.0
8,2025_01_ARI_NO,1,ARI,NO,J.Conner,0,1.0
9,2025_01_ARI_NO,1,ARI,NO,J.Conner,0,1.0


In [26]:
team_targets = (
    opportunities
    .groupby(["game_id", "posteam"])["target"]
    .sum()
    .reset_index(name="team_targets")
)

team_targets.head()

,game_id,posteam,team_targets
0,2025_01_ARI_NO,ARI,29
1,2025_01_ARI_NO,NO,41
2,2025_01_BAL_BUF,BAL,19
3,2025_01_BAL_BUF,BUF,48
4,2025_01_CAR_JAX,CAR,34


In [27]:
player_targets = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])["target"]
    .sum()
    .reset_index(name="player_targets")
)

player_targets.head()

,game_id,posteam,player_id,player_name,player_targets
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,4
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,1
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,9


In [28]:
player_targets = player_targets.merge(
    team_targets,
    on=["game_id", "posteam"],
    how="left"
)

player_targets["target_share"] = (
    player_targets["player_targets"] / player_targets["team_targets"]
)

player_targets[
    ["game_id", "posteam", "player_name", "player_targets", "team_targets", "target_share"]
].head()

,game_id,posteam,player_name,player_targets,team_targets,target_share
0,2025_01_ARI_NO,ARI,J.Conner,4,29,0.137931
1,2025_01_ARI_NO,ARI,Z.Jones,1,29,0.034483
2,2025_01_ARI_NO,ARI,K.Murray,0,29,0.000000
3,2025_01_ARI_NO,ARI,G.Dortch,1,29,0.034483
4,2025_01_ARI_NO,ARI,T.McBride,9,29,0.310345


In [29]:
team_red_zone = (
    opportunities
    .groupby(["game_id", "posteam"])["red_zone"]
    .sum()
    .reset_index(name="team_red_zone_opportunities")
)

team_red_zone.head()

,game_id,posteam,team_red_zone_opportunities
0,2025_01_ARI_NO,ARI,9
1,2025_01_ARI_NO,NO,8
2,2025_01_BAL_BUF,BAL,3
3,2025_01_BAL_BUF,BUF,21
4,2025_01_CAR_JAX,CAR,4


In [30]:
player_red_zone = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])["red_zone"]
    .sum()
    .reset_index(name="player_red_zone_opportunities")
)

player_red_zone.head()

,game_id,posteam,player_id,player_name,player_red_zone_opportunities
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,3
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,0
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,1
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,0
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,1


In [31]:
player_red_zone = player_red_zone.merge(
    team_red_zone,
    on=["game_id", "posteam"],
    how="left"
)

player_red_zone["red_zone_share"] = (
    player_red_zone["player_red_zone_opportunities"]
    / player_red_zone["team_red_zone_opportunities"]
)

player_red_zone[
    [
        "game_id",
        "posteam",
        "player_name",
        "player_red_zone_opportunities",
        "team_red_zone_opportunities",
        "red_zone_share"
    ]
].head()

,game_id,posteam,player_name,player_red_zone_opportunities,team_red_zone_opportunities,red_zone_share
0,2025_01_ARI_NO,ARI,J.Conner,3,9,0.333333
1,2025_01_ARI_NO,ARI,Z.Jones,0,9,0.000000
2,2025_01_ARI_NO,ARI,K.Murray,1,9,0.111111
3,2025_01_ARI_NO,ARI,G.Dortch,0,9,0.000000
4,2025_01_ARI_NO,ARI,T.McBride,1,9,0.111111


In [32]:
team_goal_line = (
    opportunities
    .groupby(["game_id", "posteam"])["goal_line"]
    .sum()
    .reset_index(name="team_goal_line_opportunities")
)

team_goal_line.head()

,game_id,posteam,team_goal_line_opportunities
0,2025_01_ARI_NO,ARI,3
1,2025_01_ARI_NO,NO,2
2,2025_01_BAL_BUF,BAL,0
3,2025_01_BAL_BUF,BUF,10
4,2025_01_CAR_JAX,CAR,1


In [33]:
player_goal_line = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])["goal_line"]
    .sum()
    .reset_index(name="player_goal_line_opportunities")
)

player_goal_line.head()

,game_id,posteam,player_id,player_name,player_goal_line_opportunities
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,2
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,0
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,0
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,0


In [34]:
player_goal_line = player_goal_line.merge(
    team_goal_line,
    on=["game_id", "posteam"],
    how="left"
)

player_goal_line["goal_line_share"] = (
    player_goal_line["player_goal_line_opportunities"]
    / player_goal_line["team_goal_line_opportunities"]
)

player_goal_line[
    [
        "game_id",
        "posteam",
        "player_name",
        "player_goal_line_opportunities",
        "team_goal_line_opportunities",
        "goal_line_share"
    ]
].head()

,game_id,posteam,player_name,player_goal_line_opportunities,team_goal_line_opportunities,goal_line_share
0,2025_01_ARI_NO,ARI,J.Conner,2,3,0.666667
1,2025_01_ARI_NO,ARI,Z.Jones,0,3,0.000000
2,2025_01_ARI_NO,ARI,K.Murray,0,3,0.000000
3,2025_01_ARI_NO,ARI,G.Dortch,0,3,0.000000
4,2025_01_ARI_NO,ARI,T.McBride,0,3,0.000000


In [35]:
player_goal_line["goal_line_share"] = (
    player_goal_line["goal_line_share"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

player_goal_line["goal_line_share"].isna().sum()

np.int64(0)

In [36]:
player_features = (
    player_targets
    .merge(
        player_red_zone[
            ["game_id", "posteam", "player_id", "red_zone_share"]
        ],
        on=["game_id", "posteam", "player_id"],
        how="left"
    )
    .merge(
        player_goal_line[
            ["game_id", "posteam", "player_id", "goal_line_share"]
        ],
        on=["game_id", "posteam", "player_id"],
        how="left"
    )
)

player_features.head()

,game_id,posteam,player_id,player_name,player_targets,team_targets,target_share,red_zone_share,goal_line_share
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,4,29,0.137931,0.333333,0.666667
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1,29,0.034483,0.000000,0.000000
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0,29,0.000000,0.111111,0.000000
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,1,29,0.034483,0.000000,0.000000
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,9,29,0.310345,0.111111,0.000000


In [37]:
player_game_fp = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])["actual_fp"]
    .sum()
    .reset_index(name="actual_fp_game")
)

player_game_fp.head()

,game_id,posteam,player_id,player_name,actual_fp_game
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,14.4
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1.4
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,3.8
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,0.8
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,12.1


In [38]:
player_features = player_features.merge(
    player_game_fp[
        ["game_id", "posteam", "player_id", "actual_fp_game"]
    ],
    on=["game_id", "posteam", "player_id"],
    how="left"
)

player_features.head()

,game_id,posteam,player_id,player_name,player_targets,team_targets,target_share,red_zone_share,goal_line_share,actual_fp_game
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,4,29,0.137931,0.333333,0.666667,14.4
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1,29,0.034483,0.000000,0.000000,1.4
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0,29,0.000000,0.111111,0.000000,3.8
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,1,29,0.034483,0.000000,0.000000,0.8
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,9,29,0.310345,0.111111,0.000000,12.1


In [39]:
player_volume = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])
    .agg(
        carries=("rush_attempt", "sum"),
        targets=("target", "sum")
    )
    .reset_index()
)

player_volume.head()

,game_id,posteam,player_id,player_name,carries,targets
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,12.0,4
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,0.0,1
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,7.0,0
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,0.0,1
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,0.0,9


In [40]:
player_features = player_features.merge(
    player_volume[
        ["game_id", "posteam", "player_id", "carries", "targets"]
    ],
    on=["game_id", "posteam", "player_id"],
    how="left"
)

player_features.head()

,game_id,posteam,player_id,player_name,player_targets,team_targets,target_share,red_zone_share,goal_line_share,actual_fp_game,carries,targets
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,4,29,0.137931,0.333333,0.666667,14.4,12.0,4
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1,29,0.034483,0.000000,0.000000,1.4,0.0,1
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0,29,0.000000,0.111111,0.000000,3.8,7.0,0
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,1,29,0.034483,0.000000,0.000000,0.8,0.0,1
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,9,29,0.310345,0.111111,0.000000,12.1,0.0,9


In [41]:
player_air_yards = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])
    .agg(
        total_air_yards=("air_yards", "sum"),
        avg_air_yards=("air_yards", "mean")
    )
    .reset_index()
)

player_air_yards.head()

,game_id,posteam,player_id,player_name,total_air_yards,avg_air_yards
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,-15.0,-3.750000
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,2.0,2.000000
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0.0,NaN
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,-2.0,-2.000000
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,35.0,3.888889


In [42]:
player_features = player_features.merge(
    player_air_yards[
        ["game_id", "posteam", "player_id", "total_air_yards", "avg_air_yards"]
    ],
    on=["game_id", "posteam", "player_id"],
    how="left"
)

player_features.head()

,game_id,posteam,player_id,player_name,player_targets,team_targets,target_share,red_zone_share,goal_line_share,actual_fp_game,carries,targets,total_air_yards,avg_air_yards
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,4,29,0.137931,0.333333,0.666667,14.4,12.0,4,-15.0,-3.750000
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1,29,0.034483,0.000000,0.000000,1.4,0.0,1,2.0,2.000000
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0,29,0.000000,0.111111,0.000000,3.8,7.0,0,0.0,NaN
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,1,29,0.034483,0.000000,0.000000,0.8,0.0,1,-2.0,-2.000000
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,9,29,0.310345,0.111111,0.000000,12.1,0.0,9,35.0,3.888889


In [43]:
player_features["total_air_yards"] = player_features["total_air_yards"].fillna(0)
player_features["avg_air_yards"] = player_features["avg_air_yards"].fillna(0)

player_features[
    ["player_name", "targets", "total_air_yards", "avg_air_yards"]
].head()

,player_name,targets,total_air_yards,avg_air_yards
0,J.Conner,4,-15.0,-3.750000
1,Z.Jones,1,2.0,2.000000
2,K.Murray,0,0.0,0.000000
3,G.Dortch,1,-2.0,-2.000000
4,T.McBride,9,35.0,3.888889


In [44]:
player_features["opportunities"] = (
    player_features["carries"] + player_features["targets"]
)

player_features[
    ["player_name", "carries", "targets", "opportunities"]
].head()

,player_name,carries,targets,opportunities
0,J.Conner,12.0,4,16.0
1,Z.Jones,0.0,1,1.0
2,K.Murray,7.0,0,7.0
3,G.Dortch,0.0,1,1.0
4,T.McBride,0.0,9,9.0


In [45]:
player_field_position = (
    opportunities
    .groupby(["game_id", "posteam", "player_id", "player_name"])
    .agg(
        avg_yardline_100=("yardline_100", "mean")
    )
    .reset_index()
)

player_field_position.head()

,game_id,posteam,player_id,player_name,avg_yardline_100
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,54.187500
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,37.000000
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,46.428571
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,30.000000
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,55.444444


In [46]:
player_features = player_features.merge(
    player_field_position[
        ["game_id", "posteam", "player_id", "avg_yardline_100"]
    ],
    on=["game_id", "posteam", "player_id"],
    how="left"
)

player_features.head()

,game_id,posteam,player_id,player_name,player_targets,team_targets,target_share,red_zone_share,goal_line_share,actual_fp_game,carries,targets,total_air_yards,avg_air_yards,opportunities,avg_yardline_100
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,4,29,0.137931,0.333333,0.666667,14.4,12.0,4,-15.0,-3.750000,16.0,54.187500
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,1,29,0.034483,0.000000,0.000000,1.4,0.0,1,2.0,2.000000,1.0,37.000000
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0,29,0.000000,0.111111,0.000000,3.8,7.0,0,0.0,0.000000,7.0,46.428571
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,1,29,0.034483,0.000000,0.000000,0.8,0.0,1,-2.0,-2.000000,1.0,30.000000
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,9,29,0.310345,0.111111,0.000000,12.1,0.0,9,35.0,3.888889,9.0,55.444444


In [47]:
training_data = player_features[
    [
        "game_id",
        "posteam",
        "player_id",
        "player_name",
        "target_share",
        "red_zone_share",
        "goal_line_share",
        "carries",
        "targets",
        "opportunities",
        "total_air_yards",
        "avg_air_yards",
        "avg_yardline_100",
        "actual_fp_game"
    ]
].copy()

training_data.head()

,game_id,posteam,player_id,player_name,target_share,red_zone_share,goal_line_share,carries,targets,opportunities,total_air_yards,avg_air_yards,avg_yardline_100,actual_fp_game
0,2025_01_ARI_NO,ARI,00-0033553,J.Conner,0.137931,0.333333,0.666667,12.0,4,16.0,-15.0,-3.750000,54.187500,14.4
1,2025_01_ARI_NO,ARI,00-0033891,Z.Jones,0.034483,0.000000,0.000000,0.0,1,1.0,2.0,2.000000,37.000000,1.4
2,2025_01_ARI_NO,ARI,00-0035228,K.Murray,0.000000,0.111111,0.000000,7.0,0,7.0,0.0,0.000000,46.428571,3.8
3,2025_01_ARI_NO,ARI,00-0035500,G.Dortch,0.034483,0.000000,0.000000,0.0,1,1.0,-2.0,-2.000000,30.000000,0.8
4,2025_01_ARI_NO,ARI,00-0037744,T.McBride,0.310345,0.111111,0.000000,0.0,9,9.0,35.0,3.888889,55.444444,12.1


In [48]:
training_data.to_parquet(
    "../../data/training_2025_player_games.parquet",
    index=False
)